# Camada SILVER — Limpeza e Enriquecimento
Lê da Bronze, corrige tipos, remove nulos e junta as tabelas.

In [1]:
import sys
sys.path.insert(0, '/opt/spark/work-dir')
from tools.spark_session import get_spark
from pyspark.sql import functions as F
from pyspark.sql.types import DateType

spark = get_spark('Silver - Transformação')
spark.sparkContext.setLogLevel('WARN')

BRONZE = '/opt/spark/work-dir/warehouse/bronze'
SILVER = '/opt/spark/work-dir/warehouse/silver'

26/06/18 16:31:18 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


In [2]:
# Clientes — corrige tipos e filtra nulos
clientes = (
    spark.read.format('delta').load(f'{BRONZE}/clientes')
    .withColumn('data_cadastro', F.col('data_cadastro').cast(DateType()))
    .withColumn('ativo', F.col('ativo').cast('boolean'))
    .filter(F.col('cliente_id').isNotNull())
)
clientes.show(5)
print('Clientes:', clientes.count())

26/06/18 16:32:09 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
                                                                                

+----------+--------------------+-------+------------+-------------+-----+-------------+
|cliente_id|                nome| regiao|      cidade|data_cadastro|ativo|score_credito|
+----------+--------------------+-------+------------+-------------+-----+-------------+
|         1|        Brenda Alves|    Sul|    Curitiba|   2024-01-30| true|        456.2|
|         2|Sra. Isabelly Câmara|    Sul|   Joinville|   2022-03-31| true|        322.2|
|         3|          Cauã Rocha|    Sul|Porto Alegre|   2022-08-27| true|        318.6|
|         4|  Dra. Aurora Pastor|Sudeste|      Santos|   2023-03-06| true|        712.5|
|         5|   Ana Beatriz Alves|    Sul|Porto Alegre|   2023-12-16| true|        494.5|
+----------+--------------------+-------+------------+-------------+-----+-------------+
only showing top 5 rows


Clientes: 100000


In [3]:
# Produtos — adiciona coluna 'disponivel'
produtos = (
    spark.read.format('delta').load(f'{BRONZE}/produtos')
    .withColumn('preco',     F.col('preco').cast('double'))
    .withColumn('estoque',   F.col('estoque').cast('integer'))
    .withColumn('disponivel', F.col('estoque') > 0)
)
produtos.show(5)
print('Produtos:', produtos.count())

+----------+-----------+----------+-------+-------+-------+-------------+----------+
|produto_id|       nome| categoria|  preco|estoque|peso_kg|fornecedor_id|disponivel|
+----------+-----------+----------+-------+-------+-------+-------------+----------+
|         1|Jaqueta 996|    Roupas|4627.57|    288|    4.8|           19|      true|
|         2|  Molho 670| Alimentos|4907.62|    232|  22.65|           13|      true|
|         3| Açúcar 435| Alimentos|1323.85|    181|  29.12|           46|      true|
|         4|   Capa 179|Automotivo|2220.72|    225|  21.63|           36|      true|
|         5| Lençol 244|      Casa|2806.77|     56|    9.7|           50|      true|
+----------+-----------+----------+-------+-------+-------+-------------+----------+
only showing top 5 rows


Produtos: 1000


In [4]:
# Vendas — filtra CONCLUIDO, calcula valor_total e junta dimensões
vendas = (
    spark.read.format('delta').load(f'{BRONZE}/vendas')
    .withColumn('data_venda',   F.col('data_venda').cast(DateType()))
    .withColumn('quantidade',   F.col('quantidade').cast('integer'))
    .withColumn('desconto_pct', F.col('desconto_pct').cast('double'))
    .filter(F.col('status') == 'CONCLUIDO')
    .join(produtos.select('produto_id', 'preco', 'nome', 'categoria'), 'produto_id')
    .join(clientes.select('cliente_id', 'regiao'), 'cliente_id')
    .withColumn(
        'valor_total',
        F.round(F.col('preco') * F.col('quantidade') * (1 - F.col('desconto_pct') / 100), 2)
    )
    .withColumn('ano_mes', F.date_format('data_venda', 'yyyy-MM'))
)
vendas.select('venda_id', 'nome', 'categoria', 'regiao', 'quantidade', 'valor_total', 'ano_mes').show(10)
print('Vendas CONCLUIDAS:', vendas.count())

+--------+-----------------+----------+------------+----------+-----------+-------+
|venda_id|             nome| categoria|      regiao|quantidade|valor_total|ano_mes|
+--------+-----------------+----------+------------+----------+-----------+-------+
|       1|     Película 493|Automotivo|     Sudeste|         7|   12495.04|2022-07|
|       4|Princípios de 525|    Livros|Centro-Oeste|         8|   14275.01|2022-02|
|       5|      Arte de 268|    Livros|    Nordeste|         8|   25152.48|2024-01|
|       6|     Camiseta 965|    Roupas|Centro-Oeste|         5|   22321.65|2023-09|
|       7|      Perfume 547|    Beleza|     Sudeste|        10|    2728.75|2022-09|
|       8|      Jaqueta 855|    Roupas|     Sudeste|         5|      469.7|2022-01|
|      10|         Capa 562|Automotivo|       Norte|         6|   18813.71|2023-09|
|      11|     Camiseta 965|    Roupas|     Sudeste|         4|   17857.32|2023-07|
|      14|  Tapete Auto 352|Automotivo|       Norte|         9|   16871.69|2

Vendas CONCLUIDAS: 599930


In [5]:
# Persiste na Silver
for name, df in [('clientes', clientes), ('produtos', produtos), ('vendas', vendas)]:
    df.write.format('delta').mode('overwrite').save(f'{SILVER}/{name}')
    print(f'[silver/{name}] salvo — {df.count()} linhas')

[silver/clientes] salvo — 100000 linhas


[silver/produtos] salvo — 1000 linhas


[silver/vendas] salvo — 599930 linhas
